In [1]:
# Ant Colony Optimization for Traveling Salesman Problem (TSP)

# This notebook demonstrates a simple, explainable implementation of the Ant Colony Optimization (ACO) algorithm to solve the TSP.
# The code is structured for clarity and step-by-step explanation, suitable for practical exams and oral explanation.

import numpy as np
import random

# 1. Define the distance matrix (distances between cities)
# Example: 4 cities with symmetric distances
# You can change this matrix for more cities or different distances

distance_matrix = np.array(
    [[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]]
)

# 2. Set ACO parameters
num_ants = 10
num_iterations = 50
evaporation_rate = 0.5
pheromone_constant = 1.0
heuristic_constant = 1.0

# 3. Initialize pheromone and visibility matrices
num_cities = len(distance_matrix)
pheromone = np.ones((num_cities, num_cities))  # Initial pheromone
# Avoid division by zero for diagonal
visibility = np.zeros_like(distance_matrix, dtype=float)
for i in range(num_cities):
    for j in range(num_cities):
        if i != j:
            visibility[i][j] = 1 / distance_matrix[i][j]

# 4. Main ACO loop
best_route = None
shortest_distance = float("inf")

for iteration in range(num_iterations):
    ant_routes = []
    for ant in range(num_ants):
        current_city = random.randint(0, num_cities - 1)
        visited_cities = [current_city]
        route = [current_city]

        while len(visited_cities) < num_cities:
            probabilities = []
            for city in range(num_cities):
                if city not in visited_cities:
                    pheromone_value = pheromone[current_city][city]
                    visibility_value = visibility[current_city][city]
                    probability = (pheromone_value**pheromone_constant) * (
                        visibility_value**heuristic_constant
                    )
                    probabilities.append((city, probability))
            # Normalize probabilities
            total = sum(prob for _, prob in probabilities)
            if total == 0:
                next_city = random.choice([city for city, _ in probabilities])
            else:
                probs = [prob / total for _, prob in probabilities]
                next_city = random.choices(
                    [city for city, _ in probabilities], weights=probs
                )[0]
            route.append(next_city)
            visited_cities.append(next_city)
            current_city = next_city
        ant_routes.append(route)

    # 5. Update pheromone levels
    delta_pheromone = np.zeros((num_cities, num_cities))
    for route in ant_routes:
        route_distance = sum(
            distance_matrix[route[i]][route[(i + 1) % num_cities]]
            for i in range(num_cities)
        )
        for i in range(num_cities):
            city_a = route[i]
            city_b = route[(i + 1) % num_cities]
            delta_pheromone[city_a][city_b] += 1 / route_distance
            delta_pheromone[city_b][city_a] += 1 / route_distance  # For symmetric TSP
    pheromone = (1 - evaporation_rate) * pheromone + delta_pheromone

    # 6. Track the best route found so far
    for route in ant_routes:
        route_distance = sum(
            distance_matrix[route[i]][route[(i + 1) % num_cities]]
            for i in range(num_cities)
        )
        if route_distance < shortest_distance:
            shortest_distance = route_distance
            best_route = route

# 7. Output the result
print("Best route:", best_route)
print("Shortest distance:", shortest_distance)

# 8. Explanation cell (for oral exam)
# - The algorithm simulates ants exploring routes between cities.
# - Each ant builds a tour based on pheromone and visibility (inverse distance).
# - After all ants complete their tours, pheromone is updated: more pheromone is deposited on shorter routes.
# - Over iterations, the algorithm converges to a good (often optimal) solution.
# - Parameters can be tuned for different problem sizes and convergence speed.

Best route: [3, 1, 0, 2]
Shortest distance: 80
